# 基准终值计价：同一冻结预测、零训练

假设：H5规划只扣当次费用，可能把SOXL/现金较低的进入费用误当优势。本批只对预计期末换成SOXX的价值作数学变换，保持预测、日风险协方差、实际交易费用、5日HOLD与风险门不变。

预登记参数见[plan.json](plan.json)，推导和文献边界见[mechanism.md](mechanism.md)。父Recorder四个对象必须在反序列化之前通过完整manifest和预登记SHA核验。全部历史已复用；不因此获得生产晋升。

In [1]:
from pathlib import Path
import os, sys, json
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
root = Path.cwd().resolve()
while not (root / "AGENTS.md").is_file():
    if root == root.parent:
        raise RuntimeError("workspace root unavailable")
    root = root.parent
sys.path.insert(0, str(root))
import qlib
from qlib.data import D
from qlib.workflow import R
from IPython.display import display, Markdown
card = root / "research/benchmark_terminal_value_20261009"
plan = json.loads((card / "plan.json").read_text())
qlib.init(provider_uri=str(root / plan["source"]["provider_uri"]), region="us", kernels=1)
calendar = D.calendar(freq="day", future=False)
print({"provider_uri": plan["source"]["provider_uri"], "as_of": plan["source"]["as_of"],
       "calendar_start": str(calendar[0]), "calendar_end": str(calendar[-1]), "model_fit_budget": 0})

[72659:MainThread](2026-10-09 14:10:09,838) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[72659:MainThread](2026-10-09 14:10:09,839) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[72659:MainThread](2026-10-09 14:10:09,839) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


{'provider_uri': 'data/qlib/published_futu_affine_20260925_v1', 'as_of': '2026-09-25', 'calendar_start': '2010-03-11 00:00:00', 'calendar_end': '2026-09-25 00:00:00', 'model_fit_budget': 0}


## 原生证据工作流

先核验父Recorder及重建Dataset的逐值身份；后续所有组合在没有活跃可写Recorder的上下文里执行Qlib原生回放，再恢复同一个Recorder归档。真实净值只扣当前交易费用。

逐调仓验证使用 `x=(1-fees)*open_target_weights`：其与solver post-fee values差不超过1e-12；`q*objective = direct_terminal_value - q*robust_penalty`。数值超额floor仅改变正效应分类，原收益、点估计和置信区间保留。

In [2]:
from src.workflow.benchmark_terminal_experiment import run_benchmark_terminal_experiment
summary, tables, conclusion = run_benchmark_terminal_experiment(
    source_root=root, evidence_root=root, card_dir=card,
    recorder_uri=card / "qlib_recorder")
print({key: summary[key] for key in ("recorder_id", "parent_recorder_id", "model_fits", "native_paths",
    "native_all_pass", "raw_historical_gate_pass", "historical_gate_pass", "promotion_eligible")})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[72659:MainThread](2026-10-09 14:10:11,119) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[72659:MainThread](2026-10-09 14:10:11,120) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[72659:MainThread](2026-10-09 14:10:11,120) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


[72659:MainThread](2026-10-09 14:10:11,148) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[72659:MainThread](2026-10-09 14:10:21,166) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/legacy_soxl_execution_20260925_v1')}


[72659:MainThread](2026-10-09 14:10:21,228) INFO - qlib.workflow - [exp.py:258] - Experiment 974741764376634683 starts running ...


[72659:MainThread](2026-10-09 14:10:21,231) INFO - qlib.workflow - [recorder.py:345] - Recorder f79e065bfc4e4ec7996da5f1a5379e16 starts running under Experiment 974741764376634683 ...


[72659:MainThread](2026-10-09 14:10:23,339) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'recorder_id': 'f79e065bfc4e4ec7996da5f1a5379e16', 'parent_recorder_id': 'd46772566a2947859faee38a5d4e08b8', 'model_fits': 0, 'native_paths': 6, 'native_all_pass': True, 'raw_historical_gate_pass': False, 'historical_gate_pass': False, 'promotion_eligible': False}


In [3]:
display(tables["backtest_benchmark_comparison.csv"][["strategy", "cost_bps", "window", "relative_wealth",
    "return_pct", "max_drawdown_pct", "cvar5_pct", "turnover"]])
display(tables["valuation_audit.csv"])
display(tables["parent_control_reproduction.csv"])
display(tables["native_audit.csv"][["strategy", "cost_bps", "sessions", "status", "maximum_session_return_error"]])
display(tables["gates.csv"])
display(tables["rolling_summary.csv"])

,strategy,cost_bps,window,relative_wealth,return_pct,max_drawdown_pct,cvar5_pct,turnover
0,soxx,10.0,development,1.000000,325.747970,-47.373290,-4.416437,9.990010e-01
1,soxx,10.0,reused_diagnostic,1.000000,398.621662,-42.789189,-5.295891,0.000000e+00
2,selected_original,10.0,development,0.823950,250.794873,-41.213549,-4.373946,1.539980e+01
3,selected_original,10.0,reused_diagnostic,1.000000,398.621662,-42.789189,-5.295891,0.000000e+00
4,benchmark_terminal,10.0,development,1.117046,375.580028,-41.213549,-4.407899,2.998002e+00
5,benchmark_terminal,10.0,reused_diagnostic,1.000000,398.621662,-42.789189,-5.295891,4.937355e-13
6,soxx,25.0,development,1.000000,325.110940,-47.373290,-4.418079,9.975062e-01
7,soxx,25.0,reused_diagnostic,1.000000,398.621662,-42.789189,-5.295891,0.000000e+00
8,selected_original,25.0,development,0.824631,250.559636,-41.389645,-4.364396,1.162139e+01
9,selected_original,25.0,reused_diagnostic,1.000000,398.621662,-42.789189,-5.295891,5.756355e-14


,cost_bps,solves,maximum_objective_identity_error,maximum_post_fee_values_error,future_conversion_fees_debited,model_fits,wealth_scale,maximum_covariance_roundoff_adjustment
0,10.0,540,4.440892e-16,1.448494e-16,False,0,0.999001,1.084202e-19
1,25.0,540,4.440892e-16,1.312143e-16,False,0,0.997506,5.421011e-20


,cost_bps,maximum_difference,tolerance,exact_match,passed
0,10.0,0.0,1.000000e-10,True,True
1,25.0,0.0,1.000000e-10,True,True


,strategy,cost_bps,sessions,status,maximum_session_return_error
0,soxx,10.0,2696,passed,4.440892e-16
1,selected_original,10.0,2696,passed,2.842615e-12
2,benchmark_terminal,10.0,2696,passed,9.992007e-16
3,soxx,25.0,2696,passed,2.220446e-16
4,selected_original,25.0,2696,passed,3.780865e-12
5,benchmark_terminal,25.0,2696,passed,1.332268e-15


,cost_bps,aggregate_positive_net_log_excess,minimum_positive_signal_years,median_annual_log_excess_positive,leave_any_one_year_out_log_excess_positive,anchored_max_drawdown_not_worse_than_soxx,cvar5_not_worse_than_soxx,minimum_rolling_positive_excess_rate,stationary_block_20_lower_bound_positive,stationary_block_60_lower_bound_positive,...,raw_median_annual_log_excess_positive,raw_leave_any_one_year_out_log_excess_positive,raw_anchored_max_drawdown_not_worse_than_soxx,raw_cvar5_not_worse_than_soxx,raw_minimum_rolling_positive_excess_rate,raw_stationary_block_20_lower_bound_positive,raw_stationary_block_60_lower_bound_positive,raw_all_pass,raw_positive_years,strategy
0,10.0,True,False,False,False,True,True,False,False,False,...,False,False,True,True,False,False,False,False,1,benchmark_terminal
1,25.0,False,False,False,False,True,True,False,False,False,...,False,False,True,True,False,False,False,False,0,benchmark_terminal


,cost_bps,rolling_sessions,rolling_windows,positive_windows,positive_excess_rate,minimum_positive_excess_rate,minimum_log_excess,median_log_excess,maximum_log_excess,overlapping_windows,independent_observations,within_resolution_windows,raw_positive_windows,raw_positive_excess_rate,strategy
0,10.0,252,1511,167,0.110523,0.6,0.0,0.0,0.111687,True,False,1344,167,0.110523,benchmark_terminal
1,25.0,252,1511,0,0.000000,0.6,0.0,0.0,0.000000,True,False,1511,0,0.000000,benchmark_terminal


## 可审计产物

小型指标与原始/有效分类见 [summary.json](summary.json)、[gates.csv](gates.csv)、[raw_gates.csv](raw_gates.csv)、[valuation_audit.csv](valuation_audit.csv) 和 [parent_control_reproduction.csv](parent_control_reproduction.csv)。完整预测、路径、Positions、求解证书与终值逐笔核验都由Qlib Recorder持有，压缩对象带双SHA校验。

In [4]:
display(Markdown(conclusion))

# 基准终值计价实验

本轮没有训练模型：从已封存父Recorder读取相同年度选择预测，仅改变H5规划的终值计价。两种方案独立扣除当次真实交易费用，并通过Qlib原生持仓估值核验。

25bp结果如下；开发期为完整2016–2022信号年，后段仍是已复用历史：

- development/soxx：收益325.11%，相对SOXX财富1.000000，最大回撤-47.37%，CVaR5 -4.42%，换手0.9975。
- reused_diagnostic/soxx：收益398.62%，相对SOXX财富1.000000，最大回撤-42.79%，CVaR5 -5.30%，换手0.0000。
- development/selected_original：收益250.56%，相对SOXX财富0.824631，最大回撤-41.39%，CVaR5 -4.36%，换手11.6214。
- reused_diagnostic/selected_original：收益398.62%，相对SOXX财富1.000000，最大回撤-42.79%，CVaR5 -5.30%，换手0.0000。
- development/benchmark_terminal：收益325.11%，相对SOXX财富1.000000，最大回撤-47.37%，CVaR5 -4.42%，换手0.9975。
- reused_diagnostic/benchmark_terminal：收益398.62%，相对SOXX财富1.000000，最大回撤-42.79%，CVaR5 -5.30%，换手0.0000。

原始符号门通过：False；排除1e-8以内累计对数超额后的门通过：False。收益、点估计和置信区间保持原值；原始/有效正年及滚动胜率分别保留。

终值转换仅是规划代理，不发生未来清仓交易，也没有把假设的未来转换费用再次扣入真实净值。逐次求解的q倍目标已与直接终值减歧义惩罚核对；风险协方差、费用、初始现金、预测和持有频率不变。

年度、滚动及压力差异见小型侧表；源数据口径、16条次来源行情、连续份额、历史开盘及现金零收益假设仍有限制。估计风险预算只在调仓时约束，中间HOLD漂移单独记录。

不晋升。所有历史已被复用，本批完成不代表稳定超额目标完成。下一步依据保留的预测、实际动作、尾部与计价差异独立诊断，再登记下一机制。

Recorder：f79e065bfc4e4ec7996da5f1a5379e16；父Recorder：d46772566a2947859faee38a5d4e08b8；as_of=2026-09-25。
